In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture

data = pd.read_csv("../data/processed/data_scored.csv")

data.head()

,Jenis Kelamin,Semester saat ini,Angkatan,Departemen,Program Studi,WB1,WB2,WB3,WB4,WB5,...,CAR3,CAR4,CAR5,CAR6,CAR7,CAR8,WB_Score,ACD_Score,SOC_Score,CAR_Score
0,Laki-laki,Semester 5,2024,Departemen Teknik Elektro (DTE),S. Tr. Teknik Elektro Industri,4,4,3,3,3,...,3,3,3,4,3,3,3.666667,3.250,4.000,3.125
1,Perempuan,Semester 3,2025,Departemen Teknologi Multimedia Kreatif (DTMK),D3 Teknologi Multimedia dan Broadcasting,1,4,1,2,1,...,2,1,1,1,1,1,2.666667,2.875,3.375,1.250
2,Laki-laki,Semester 5,2024,Departemen Teknik Informatika dan Komputer (DTIK),S.Tr. Teknik Informatika,3,4,4,4,2,...,4,2,3,4,3,2,3.444444,3.375,3.375,2.750
3,Laki-laki,Semester 5,2024,Departemen Teknik Informatika dan Komputer (DTIK),S.Tr. Sains Data Terapan,3,4,4,3,4,...,4,2,3,3,3,4,3.555556,3.750,3.750,3.375
4,Laki-laki,Semester 5,2024,Departemen Teknik Informatika dan Komputer (DTIK),D3 Teknik Informatika,3,2,4,4,3,...,2,1,1,1,1,1,2.777778,3.500,4.750,1.125


In [2]:
score_cols = [
    "WB_Score",
    "ACD_Score",
    "SOC_Score",
    "CAR_Score"
]

lpa_data = data[score_cols].copy()

lpa_data.head()

,WB_Score,ACD_Score,SOC_Score,CAR_Score
0,3.666667,3.250,4.000,3.125
1,2.666667,2.875,3.375,1.250
2,3.444444,3.375,3.375,2.750
3,3.555556,3.750,3.750,3.375
4,2.777778,3.500,4.750,1.125


In [3]:
scaler = StandardScaler()

X = scaler.fit_transform(lpa_data)

X[:5]

array([[ 0.36126865, -0.16379175,  0.0999372 , -0.13129931],
       [-0.93711634, -0.67671855, -0.74952902, -2.32166639],
       [ 0.07273866,  0.00718385, -0.74952902, -0.56937273],
       [ 0.21700366,  0.52011065, -0.23984928,  0.16074963],
       [-0.79285134,  0.17815945,  1.11929666, -2.46769086]])

In [4]:
results = []

for k in range(2, 7):
    model = GaussianMixture(
        n_components=k,
        covariance_type="full",
        random_state=42
    )
    
    model.fit(X)
    
    results.append({
        "K": k,
        "AIC": model.aic(X),
        "BIC": model.bic(X),
        "Log_Likelihood": model.lower_bound_
    })

model_summary = pd.DataFrame(results)

model_summary

C:\Users\amel\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\amel\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.10_3.10.3056.0_x64__qbz5n2kfra8p0\lib\subprocess.py", line 503, in run
    with Popen(*popenargs, **kwargs) as process:
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.

,K,AIC,BIC,Log_Likelihood
0,2,1106.497203,1187.091784,-4.406025
1,3,1082.461120,1204.742554,-4.178680
2,4,1094.347178,1258.315465,-4.102801
3,5,955.876762,1161.531900,-3.394612
4,6,1060.088506,1307.430497,-3.706792


In [5]:
best_k = 3

lpa_model = GaussianMixture(
    n_components=best_k,
    covariance_type="full",
    random_state=42
)

lpa_model.fit(X)

GaussianMixture(n_components=3, random_state=42)

In [6]:
data["Profile"] = lpa_model.predict(X) + 1

probabilities = lpa_model.predict_proba(X)

for i in range(best_k):
    data[f"Profile_{i+1}_Prob"] = probabilities[:, i]

data.head()

,Jenis Kelamin,Semester saat ini,Angkatan,Departemen,Program Studi,WB1,WB2,WB3,WB4,WB5,...,CAR7,CAR8,WB_Score,ACD_Score,SOC_Score,CAR_Score,Profile,Profile_1_Prob,Profile_2_Prob,Profile_3_Prob
0,Laki-laki,Semester 5,2024,Departemen Teknik Elektro (DTE),S. Tr. Teknik Elektro Industri,4,4,3,3,3,...,3,3,3.666667,3.250,4.000,3.125,3,1.194477e-01,1.668842e-02,0.863864
1,Perempuan,Semester 3,2025,Departemen Teknologi Multimedia Kreatif (DTMK),D3 Teknologi Multimedia dan Broadcasting,1,4,1,2,1,...,1,1,2.666667,2.875,3.375,1.250,3,1.226395e-19,2.976047e-05,0.999970
2,Laki-laki,Semester 5,2024,Departemen Teknik Informatika dan Komputer (DTIK),S.Tr. Teknik Informatika,3,4,4,4,2,...,3,2,3.444444,3.375,3.375,2.750,3,5.481093e-10,8.206531e-02,0.917935
3,Laki-laki,Semester 5,2024,Departemen Teknik Informatika dan Komputer (DTIK),S.Tr. Sains Data Terapan,3,4,4,3,4,...,3,4,3.555556,3.750,3.750,3.375,3,8.109201e-03,3.478828e-01,0.644008
4,Laki-laki,Semester 5,2024,Departemen Teknik Informatika dan Komputer (DTIK),D3 Teknik Informatika,3,2,4,4,3,...,1,1,2.777778,3.500,4.750,1.125,1,9.999973e-01,1.551169e-17,0.000003


In [8]:
data.to_csv(
    "../data/processed/data_lpa.csv",
    index=False
)

model_summary.to_csv(
    "../data/output/model_summary.csv",
    index=False
)

print("Hasil LPA-GMM berhasil disimpan.")

Hasil LPA-GMM berhasil disimpan.
